In [4]:
from stormvogel import *
print("sanity")

sanity


In [2]:
%%prism grid_prism
mdp

const int N = 5; // Grid size

// Ranger and lions initial positions
const int R_X = 0; 
const int R_Y = N-1;

const int F_X = 0;
const int F_Y = 0;

const int C_X = N-1;
const int C_Y = (N-1)/2;

// Camera positions
const int C1_X = 1;
const int C1_Y = 2;
const int C2_X = (N-1)/2;
const int C2_Y = 0;
const int C3_X = N-1;
const int C3_Y = N-2;

// Cameras
global c_1 : [0..2] init 0; // camera 1, 0: nothing, 1: claw, 2: fang
global c_2: [0..2] init 0;
global c_3: [0..2] init 0;
global i_c: bool init false; // inspected claw
global i_f: bool init false; // inspected fang

// Ranger pos and decision
global r_x : [0..N-1] init R_X;
global r_y : [0..N-1] init R_Y;
global decision : [0..2] init 0;
// 0: No decision made
// 1: move
// 2: inspect

// Claw and fang positions
global c_x : [0..N-1] init C_X; // Claw tribe
global c_y : [0..N-1] init C_Y;
global f_x : [0..N-1] init F_X;
global f_y : [0..N-1] init F_Y;

module ranger_decision
    [move] (decision = 0) -> 1 : (decision' = 1);
    [inspect] ( (on_fang | on_claw | on_c1 | on_c2 | on_c3) & decision = 0) -> 1 : (decision' = 2);
    [step] (decision > 0) -> 1 : (decision' = 0); // Sync on step.
endmodule

module ranger_movement
    [step] (r_x > 0 & decision = 1) -> 0.8 : (r_x' = r_x - 1) & (r_y' = r_y)
                                     + 0.2 : (r_x' = r_x) & (r_y' = r_y);
    [step] (r_x < N-1 & decision = 1) -> 0.8: (r_x' = r_x + 1) & (r_y' = r_y)
                                     + 0.2 : (r_x' = r_x) & (r_y' = r_y);
    [step] (r_y > 0 & decision = 1) -> 0.8: (r_x' = r_x) & (r_y' = r_y - 1)
                                     + 0.2 : (r_x' = r_x) & (r_y' = r_y);
    [step] (r_y < N-1 & decision = 1) -> 0.8: (r_x' = r_x) & (r_y' = r_y + 1)
                                     + 0.2 : (r_x' = r_x) & (r_y' = r_y);
    [step] (decision = 2) -> 1 : (r_x'=r_x) & (r_y' = r_y); // Sync if inspect.
endmodule

module inspect
    // Inspect claw or fang directly
    [inspect] (on_claw) -> 0.95 : (i_c' = true)
                        + 0.05 : (i_c' = i_c);
    [inspect] (on_fang) -> 0.95 : (i_f' = true)
                        + 0.05 : (i_f' = i_f);

    // Inspect the picture of claw (only possible if neither claw nor fang is on the current position)                                     
    [inspect] (on_c1 & c_1 = 1) -> 0.6 : (i_c' = true) & (c_1' = 0)
                                 + 0.4 : (i_c' = i_c) & (c_1' = 0);                                
    [inspect] (on_c2 & c_2 = 1) -> 0.6 : (i_c' = true) & (c_2' = 0)
                                 + 0.4 : (i_c' = i_c) & (c_2' = 0);           
    [inspect] (on_c3 & c_3 = 1) -> 0.6 : (i_c' = true) & (c_3' = 0)
                                 + 0.4 : (i_c' = i_c) & (c_3' = 0);

    // Inspect the picture of fang (only possible if neither claw nor fang is on the current position)   
    [inspect] (on_c1 & c_1 = 2) -> 0.6 : (i_f' = true) & (c_1' = 0)
                                 + 0.4 : (i_f' = i_f) & (c_1' = 0);
    [inspect] (on_c2 & c_2 = 2) -> 0.6 : (i_f' = true) & (c_2' = 0)
                                 + 0.4 : (i_f' = i_f) & (c_2' = 0);
    [inspect] (on_c3 & c_3 = 2) -> 0.6 : (i_f' = true) & (c_3' = 0)
                                 + 0.4 : (i_f' = i_f) & (c_3' = 0);
    // The cases where inspect doesnt make sense are excluded from the model!
endmodule
    
module camera1
    [step] (claw_on_c1) -> 1 : (c_1' = 1);
    [step] (fang_on_c1) -> 1 : (c_1' = 2);
    [step] (! claw_on_c1 & ! fang_on_c1) -> 1 : (c_1' = c_1);
endmodule

module camera2
    [step] (claw_on_c2) -> 1 : (c_2' = 1);
    [step] (fang_on_c2) -> 1 : (c_2' = 2);
    [step] (! claw_on_c2 & ! fang_on_c2 ) -> 1 : (c_2' = c_2);
endmodule

module camera3
    [step] (claw_on_c3) -> 1 : (c_3' = 1);
    [step] (fang_on_c3) -> 1 : (c_3' = 2);
    [step] (! claw_on_c3 & ! fang_on_c3) -> 1 : (c_3' = c_3);
endmodule

formula on_fang = r_x = f_x & r_y = f_y;
formula on_claw = r_x = c_x & r_y = c_y;
formula on_c1 = r_x = C1_X & r_y = C1_Y;
formula on_c2 = r_x = C2_X & r_y = C2_Y;
formula on_c3 = r_x = C3_X & r_y = C3_Y;

formula claw_on_c1 = c_x = C1_X & c_y = C1_Y;
formula claw_on_c2 = c_x = C2_X & c_y = C2_Y;
formula claw_on_c3 = c_x = C3_X & c_y = C3_Y;

formula fang_on_c1 = f_x = C1_X & f_y = C1_Y;
formula fang_on_c2 = f_x = C2_X & f_y = C2_Y;
formula fang_on_c3 = f_x = C3_X & f_y = C3_Y;

// Claw movement possibilities
formula claw_move_left = !((f_x = c_x - 1 & (f_y = c_y | f_y = c_y + 1 | f_y = c_y - 1)) | (f_x = c_x - 2 & f_y = c_y)) & (c_x > 0) ? 1 : 0;
formula claw_move_right = !((f_x = c_x + 1 & (f_y = c_y | f_y = c_y + 1 | f_y = c_y - 1)) | (f_x = c_x + 2 & f_y = c_y)) & c_x < N - 1 ? 1 : 0;
formula claw_move_up = !((f_y = c_y - 1 & (f_x = c_x | f_x = c_x + 1 | f_x = c_x - 1)) | (f_y = c_y - 2 & f_x = c_x)) & c_y > 0 ? 1 : 0;
formula claw_move_down = !((f_y = c_y + 1 & (f_x = c_x | f_x = c_x + 1 | f_x = c_x - 1)) | (f_y = c_y + 2 & f_x = c_x)) & c_y < N - 1 ? 1 : 0;
formula claw_moves = (claw_move_left + claw_move_right + claw_move_up + claw_move_down) > 0 ? 1/(claw_move_left + claw_move_right + claw_move_up + claw_move_down) : 0;

// claw_moves = 1 / (number of directions that claw can move to). If there is no direction to move to, it is 0 instead.
module claw
    [step] (true) -> 0.12 * claw_moves * claw_move_left : (c_x' = c_x - 1) & (c_y' = c_y)
                   + 0.12 * claw_moves * claw_move_right : (c_x' = c_x + 1) & (c_y' = c_y)
                   + 0.12 * claw_moves * claw_move_up : (c_x' = c_x) & (c_y' = c_y - 1)
                   + 0.12 * claw_moves * claw_move_down: (c_x' = c_x) & (c_y' = c_y + 1)
                   + max(0.88, 1 - claw_moves): (c_x' = c_x) & (c_y' = c_y);
endmodule

// Fang movement possibilities
formula fang_move_left = !((c_x = f_x - 1 & (c_y = f_y | c_y = f_y + 1 | c_y = f_y - 1)) | (c_x = f_x - 2 & c_y = f_y)) & (f_x > 0) ? 1 : 0;
formula fang_move_right = !((c_x = f_x + 1 & (c_y = f_y | c_y = f_y + 1 | c_y = f_y - 1)) | (c_x = f_x + 2 & c_y = f_y) ) & f_x < N - 1 ? 1 : 0;
formula fang_move_up = !((c_y = f_y - 1 & (c_x = f_x | c_x = f_x + 1 | c_x = f_x - 1)) | (c_y = f_y - 2 & c_x = f_x) ) & f_y > 0 ? 1 : 0;
formula fang_move_down = !( (c_y = f_y + 1 & (c_x = f_x | c_x = f_x + 1 | c_x = f_x - 1)) | (c_y = f_y + 2 & c_x = f_x) ) & f_y < N - 1 ? 1 : 0;
formula fang_moves = (fang_move_left + fang_move_right + fang_move_up + fang_move_down) > 0 ? 1/(fang_move_left + fang_move_right + fang_move_up + fang_move_down) : 0;

module fang
     [step] (true) -> 0.12 * fang_moves * fang_move_left : (f_x' = f_x - 1) & (f_y' = f_y)
                   + 0.12 * fang_moves * fang_move_right : (f_x' = f_x + 1) & (f_y' = f_y)
                   + 0.12 * fang_moves * fang_move_up : (f_x' = f_x) & (f_y' = f_y - 1)
                   + 0.12 * fang_moves * fang_move_down: (f_x' = f_x) & (f_y' = f_y + 1)
                   + max(0.88, 1 - fang_moves): (f_x' = f_x) & (f_y' = f_y);
endmodule

rewards "steps"
    [step] true : 1; //whenever a step is taken, add 1 to the timesteps reward.
endrewards

// assign labels to inspected fang/claw states.
label "inspected_fang" = i_f;
label "inspected_claw" = i_c;
label "inspected_both" = i_f & i_c;
label "overlap" = f_x = c_x & f_y = c_y;
label "on_fang" = f_x = r_x & f_y = r_y;
label "on_claw" = c_x = r_x & c_y = r_y;
label "doubt" = decision = 0;
label "move" = decision = 1;
label "inspect" = decision = 2;
label "c1_claw" = c_1 = 1;
label "c1_fang" = c_1 = 2;
label "c2_claw" = c_2 = 1;
label "c2_fang" = c_2 = 2;
label "c3_claw" = c_3 = 1;
label "c3_fang" = c_3 = 2;
label "cx1" = c_x = 1;
label "cx2" = c_x = 2;
label "cy1" = c_y = 1;
label "cy2" = c_y = 2;
label "onc1" = r_x = C1_X & r_y = C1_Y;
label "onc2" = r_x = C2_X & r_y = C2_Y;
label "onc3" = r_x = C3_X & r_y = C3_Y;

In [ ]:
# Convert to Stormvogel and visualize if it is not too big. 
# This takes a long time for the 5x5 model.
import stormpy
sp_model = stormpy.build_model(grid_prism)
print(sp_model)
no_states = sp_model.transition_matrix.nr_columns
if no_states < 50:
    sv_model = mapping.stormpy_to_stormvogel(sp_model)
    vis = show(sv_model, show_editor=True, layout=Layout("lion.json"))

## Exercise 1
Model checking property "1": Pmax=? [G !"overlap"] ...

Result (for initial states): 1


## Exercise 2.1: The best effort probability that the health of the entire lion population is estimated in one day

Model checking property "1": Pmax=? [true U<=28 "inspected_both"] ...

Result (for initial states): 0.6402461625


## Exercise 2.2: Which tribe is easiest to go after first?

### Tribe 1: Fang (top-left)

Model checking property "1": Rmin=? [F "inspected_fang"] ...

Result (for initial states): 5.220809785

### Tribe 2: Claw (middle-right)

Model checking property "1": Rmin=? [F "inspected_claw"] ...

Result (for initial states): 7.52024229

### Conclusion

Fang is easiest to go after first, with a minimum reward of 5.22, meaning it can be reached within 5.22 hours, which is 7.52 - 5.22 = 2.30 hours quicker than Claw.

## Exercise 3.1

### Fang

#### Camera 1

Model checking property "1": Pmax=? [true U<=28 "c1_fang"] ...
Result (for initial states): 0.02889103201

#### Camera 2

Model checking property "1": Pmax=? [true U<=28 "c2_fang"] ...
Result (for initial states): 0.08497620179

#### Camera 3

Model checking property "1": Pmax=? [true U<=28 "c3_fang"] ...
Result (for initial states): 4.922475384e-07

### Claw

#### Camera 1

Model checking property "1": Pmax=? [true U<=28 "c1_claw"] ...
Result (for initial states): 0.004120408477

#### Camera 2

Model checking property "1": Pmax=? [true U<=28 "c2_claw"] ...
Result (for initial states): 0.002509828426

#### Camera 3

Model checking property "1": Pmax=? [true U<=28 "c3_claw"] ...
Result (for initial states): 0.2881073003

## Exercise 3.2

Camera 3 currently has the best placement. 

Sector (3,3), one space left of the middle-right tribe (Claw), would likely be best, since it would be closer to the rangers, and to the other lion tribe.


---

In [ ]:
# Exercise 1
!storm --prism ./lions.prism --engine dd -prop "Pmax=? [G ! \"overlap\"]"

In [ ]:
# Exercise 2.1
!storm --prism ./lions.prism --engine dd -prop "Pmax=? [F<=28 \"inspected_both\"]"

In [ ]:
# Exercise 2.2 (Fang)
!storm --prism ./lions.prism --engine sparse -prop "Rmin=? [F \"inspected_fang\"]"

In [ ]:
# Exercise 2.2 (Claw)
!storm --prism ./lions.prism --engine sparse -prop "Rmin=? [F \"inspected_claw\"]"

In [ ]:
# Exercise 3.1 (Cam 1, Fang)
!storm --prism ./lions.prism --engine dd -prop "Pmax=? [F<=28 \"c1_fang\"]"

In [ ]:
# Exercise 3.1 (Cam 2, Fang)
!storm --prism ./lions.prism --engine dd -prop "Pmax=? [F<=28 \"c2_fang\"]"

In [ ]:
# Exercise 3.1 (Cam 3, Fang)
!storm --prism ./lions.prism --engine dd -prop "Pmax=? [F<=28 \"c3_fang\"]"

In [ ]:
# Exercise 3.1 (Cam 1, Claw)
!storm --prism ./lions.prism --engine dd -prop "Pmax=? [F<=28 \"c1_claw\"]"

In [ ]:
# Exercise 3.1 (Cam 2, Claw)
!storm --prism ./lions.prism --engine dd -prop "Pmax=? [F<=28 \"c2_claw\"]"

In [ ]:
# Exercise 3.1 (Cam 3, Claw)
!storm --prism ./lions.prism --engine dd -prop "Pmax=? [F<=28 \"c3_claw\"]"